# Music Annotation Project V8.6 - Notebook 46R6
## POST-OUTPUT SECTION ONLY

**Status:** development-only scientific audit. This notebook has no production or promotion authority.

46R6 carries forward the 46R5 experiment while tightening resumability and auditability. It retains the frozen M0-M4 mask arms, the pre-registered restorer arms, source-group held-out learned restoration, exact outside-mask immutability, whole-page candidate assembly, post-output truth scoring, paired page-cluster inference and a curated final upload bundle.




In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy pyarrow scikit-image matplotlib 

In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, random, shutil, sys, platform, zipfile, gc, time, os
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt, torch
from skimage.restoration import inpaint as sk_inpaint
from skimage.metrics import structural_similarity as ssim
from torch import nn
from torch.utils.data import Dataset, DataLoader

ROOT = Path('/content/drive/MyDrive/MusicAnnotationProject')
N45_ROOT = ROOT/'observed_print_classifier_v8_6_notebook45'
OUTROOT = ROOT/'observed_print_classifier_v8_6_notebook46'

SOURCE_N45_RUN_OVERRIDE = None
RESUME_RUN_OVERRIDE = None  # Strongly recommended: paste the exact crashed run directory here.
AUTO_RESUME_LATEST_INCOMPLETE = True
ALLOW_NEW_RUN_IF_NONE_FOUND = True

SEED = 46001
BOOT = 10000
PATCH = 128
EPOCHS = 8
BATCH = 16
ENABLE_LEARNED_RESTORER = True
VALIDATE_EXISTING_OUTPUT_SHA = False  # Shape and immutability are always checked; SHA is recorded.
VALIDATE_EXISTING_CANDIDATE_SHA = False  # Optional expensive hash check against prior candidate inventory.
REUSE_VALID_WHOLE_PAGE_CANDIDATES = True
SAVE_PROGRESS_EVERY_REGION = True

# Post-output-only notebook: reconstruction outputs already exist, so downstream scoring runs on CPU.
DEVICE = 'cpu'


def candidate_resume_runs(root):
    required = ['source_record.json', 'restorer_pre_registration.json']
    hits = []
    if root.exists():
        for p in root.iterdir():
            if p.is_dir() and all((p/r).exists() for r in required):
                # A completed run is still safe to reopen, but incomplete runs are preferred.
                complete = (p/'v8_6_verdict.json').exists() and (p/'tables/paired_region_metrics.csv').exists()
                hits.append((complete, p.stat().st_mtime, p))
    return [p for complete, _, p in sorted(hits, key=lambda x:(x[0], -x[1]))]

if RESUME_RUN_OVERRIDE is not None:
    OUT = Path(RESUME_RUN_OVERRIDE)
    if not OUT.exists():
        raise RuntimeError(f'RESUME_RUN_OVERRIDE does not exist: {OUT}')
    RESUME_MODE = True
elif AUTO_RESUME_LATEST_INCOMPLETE:
    hits = candidate_resume_runs(OUTROOT)
    OUT = hits[0] if hits else None
    RESUME_MODE = OUT is not None
else:
    OUT = None
    RESUME_MODE = False

if OUT is None:
    if not ALLOW_NEW_RUN_IF_NONE_FOUND:
        raise RuntimeError('No resumable Notebook 46 run found. Set RESUME_RUN_OVERRIDE.')
    run = datetime.now().strftime('%Y%m%d_%H%M%S') + '_v8_6_notebook46R'
    OUT = OUTROOT/run
    RESUME_MODE = False

TABLES=OUT/'tables'; OUTPUTS=OUT/'outputs'; MODELS=OUT/'models'; REVIEW=OUT/'visual_audit'; AUDIT=OUT/'audit'; UPLOAD=OUT/'upload'; WHOLE=OUT/'whole_page_candidates'
for p in [OUT,TABLES,OUTPUTS,MODELS,REVIEW,AUDIT,UPLOAD,WHOLE]:
    p.mkdir(parents=True, exist_ok=True)

print('Mode:', 'RESUME' if RESUME_MODE else 'NEW RUN')
print('Output:', OUT)
print('Device:', DEVICE, torch.cuda.get_device_name(0) if DEVICE == 'cuda' else 'CPU')

In [ ]:
# 46R6 contract fingerprint and atomic I/O helpers.
NOTEBOOK_VERSION = "46R6"
CONTRACT_FIELDS = {
    "notebook_version": NOTEBOOK_VERSION,
    "programme": "V8.6",
    "mask_arms": ["M0", "M1", "M2", "M3", "M4"],
    "development_only": True,
    "outside_mask_immutable": True,
    "truth_scoring_post_output_only": True,
}

def canonical_json_bytes(obj):
    return json.dumps(obj, sort_keys=True, separators=(",", ":"), ensure_ascii=True).encode("utf-8")

def contract_fingerprint(extra=None):
    payload = dict(CONTRACT_FIELDS)
    if extra:
        payload["runtime"] = extra
    return hashlib.sha256(canonical_json_bytes(payload)).hexdigest()

def atomic_write_bytes(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    with open(tmp, "wb") as f:
        f.write(payload)
        f.flush()
        os.fsync(f.fileno())
    tmp.replace(path)

def atomic_write_json(path, obj):
    atomic_write_bytes(path, json.dumps(obj, indent=2, sort_keys=True).encode("utf-8"))

def validate_checkpoint_record(record, expected_fingerprint, required_paths=()):
    if not isinstance(record, dict):
        return False, "record_not_dict"
    if record.get("contract_fingerprint") != expected_fingerprint:
        return False, "contract_fingerprint_mismatch"
    if record.get("status") != "complete":
        return False, "checkpoint_not_complete"
    for p in required_paths:
        if not Path(p).exists():
            return False, f"missing:{p}"
    return True, "valid"

RUN_CONTRACT_FINGERPRINT = contract_fingerprint()
print("46R6 contract fingerprint:", RUN_CONTRACT_FINGERPRINT)


In [ ]:
def sha(p):
    h=hashlib.sha256()
    with open(p,'rb') as f:
        for b in iter(lambda:f.read(1048576),b''):
            h.update(b)
    return h.hexdigest()

def atomic_json(path, value):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    tmp.write_text(json.dumps(value,indent=2,sort_keys=True,default=str))
    tmp.replace(path)

def atomic_csv(frame, path):
    path=Path(path); tmp=Path(str(path)+'.tmp')
    frame.to_csv(tmp,index=False); tmp.replace(path)

def latest(root,req):
    hits=[p for p in Path(root).rglob('*') if p.is_dir() and all((p/r).exists() for r in req)] if Path(root).exists() else []
    return max(hits,key=lambda p:p.stat().st_mtime) if hits else None

REQ45=['mask_lock.json','notebook45_verdict.json','source_record.json','tables/region_manifest.parquet','tables/mask_readiness_metrics.csv']
N45=Path(SOURCE_N45_RUN_OVERRIDE) if SOURCE_N45_RUN_OVERRIDE else latest(N45_ROOT,REQ45)
if N45 is None:
    raise RuntimeError('No complete Notebook 45 run found.')

lock=json.loads((N45/'mask_lock.json').read_text())
verdict45=json.loads((N45/'notebook45_verdict.json').read_text())
source=json.loads((N45/'source_record.json').read_text())
regions=pd.read_parquet(N45/'tables/region_manifest.parquet')

record={
    'notebook45':str(N45),
    'mask_lock_sha256':sha(N45/'mask_lock.json'),
    'mask_lock_id':lock['lock_id'],
    'source_record_sha256':sha(N45/'source_record.json'),
    'notebook45_ready':verdict45.get('ready_for_crossover'),
}

source_record_path=OUT/'source_record.json'
if source_record_path.exists():
    previous=json.loads(source_record_path.read_text())
    for key in ['mask_lock_sha256','mask_lock_id','source_record_sha256']:
        if previous.get(key)!=record.get(key):
            raise RuntimeError(f'Resume blocked: {key} differs from the crashed run.')
else:
    atomic_json(source_record_path,record)


print(json.dumps(record,indent=2))
print('Regions:',len(regions),'Pages:',regions.page.nunique(),'Source groups:',regions.source_group.nunique())

In [ ]:
RESTORERS=['R0_unresolved','R1_biharmonic','R2_telea','R3_learned_unet']
PRE={
    'version':'8.6-notebook46R-resumable-development',
    'mask_lock_id':lock['lock_id'],
    'restorers':RESTORERS,
    'epochs':EPOCHS,
    'patch':PATCH,
    'seed':SEED,
    'clean_truth_role':'training targets on permitted source groups and post-output scoring only',
    'best_of_n_selection':False,
    'resume_contract':{
        'reuse_valid_checkpoints':True,
        'reuse_valid_outputs':True,
        'recompute_missing_or_invalid_only':True,
        'progress_written_after_each_region':True,
    },
}
pre_path=OUT/'restorer_pre_registration.json'
if pre_path.exists():
    old=json.loads(pre_path.read_text())
    immutable_keys=['mask_lock_id','restorers','epochs','patch','seed']
    for key in immutable_keys:
        if old.get(key)!=PRE.get(key):
            raise RuntimeError(f'Resume blocked: pre-registration field changed: {key}')
else:
    atomic_json(pre_path,PRE)

source_v82=Path(source['source_v82'])
clean_map={p.stem:p for p in (ROOT/'source'/'clean').iterdir() if p.suffix.lower() in {'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}}
page_obs={p:cv2.imread(str(source_v82/'whole_pages'/p/'observed.png'),0) for p in regions.page.unique()}
page_clean={p:cv2.imread(str(clean_map[p]),0) for p in regions.page.unique()}
print('Pre-registration verified. Existing run may be resumed.')

In [ ]:
# Restore post-output runtime state removed with the original whole-page assembly cells.
# This reconstructs page_auth exactly as 46R5 did, from the V7.5 authoritative mask cache.
source45=json.loads((N45/'source_record.json').read_text())
v75=Path(source45['v75'])
v75_status=pd.read_csv(v75/'tables/cache_status.csv')
v75_cache={str(r.page):Path(r.cache) for _,r in v75_status.iterrows()}
page_auth={}
for page in sorted(regions.page.unique()):
    if page not in v75_cache:
        raise RuntimeError(f'{page}: V7.5 cache missing while restoring page_auth')
    with np.load(v75_cache[page]) as z:
        if 'mask' not in z.files:
            raise RuntimeError(f'{page}: V7.5 cache missing authoritative mask')
        auth=z['mask']>0
    obs=page_obs[page]
    if obs is None or auth.shape!=obs.shape:
        raise RuntimeError(f'{page}: observed/authoritative-mask shape mismatch')
    page_auth[page]=auth

candidate_inventory_path=TABLES/'whole_page_candidate_inventory.csv'
if not candidate_inventory_path.exists():
    raise RuntimeError(f'Missing upstream whole-page candidate inventory: {candidate_inventory_path}')
print('Restored page_auth for',len(page_auth),'pages; whole-page candidate inventory present.')


## Post-output truth scoring

In [ ]:
# Accelerated, resume-safe post-output truth scoring.
# Scientific outputs are unchanged: the optimisation caches clean-derived features,
# computes metrics only on a sufficiently padded crop around each region, groups work
# by region/arm, and checkpoints less frequently.

SCORING_CROP_PAD = 24       # adaptiveThreshold radius 20; SSIM radius 3; tolerant-F1 radius 1
CHECKPOINT_NEW_ROWS = 2000  # avoids repeatedly rewriting a large CSV every 100 rows
EQUIVALENCE_SAMPLE = 16     # compare accelerated recomputation with rows already produced by the original cell

def ink(im):
    return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,
                                 cv2.THRESH_BINARY_INV,41,15)>0

def edges(im):
    return cv2.Canny(im,70,170)>0

def f1_from_features(ref,pred,reg,tol=1):
    ref=ref&reg; pred=pred&reg
    if not ref.any(): return np.nan,np.nan,np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    p=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    q=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return float(p),float(q),float(2*p*q/max(p+q,1e-12))

def crop_box(mask,pad=SCORING_CROP_PAD):
    ys,xs=np.where(mask)
    if not len(ys): return None
    h,w=mask.shape
    return (max(0,int(ys.min())-pad), min(h,int(ys.max())+pad+1),
            max(0,int(xs.min())-pad), min(w,int(xs.max())+pad+1))

def score_one_fast(clean,clean_ink,clean_edges,pred,mask,box,meta):
    if box is None or not mask.any():
        return {**meta,'pixels':0,'mae':np.nan,'ssim_local':np.nan,
                'ink_precision':np.nan,'ink_recall':np.nan,'ink_f1':np.nan,'edge_f1':np.nan,
                'false_inserted_ink':0,'missing_ink':0}
    y0,y1,x0,x1=box
    c=clean[y0:y1,x0:x1]
    p=pred[y0:y1,x0:x1]
    m=mask[y0:y1,x0:x1]
    ci=clean_ink
    ce=clean_edges
    pi=ink(p)
    pe=edges(p)
    diff=np.abs(c.astype(np.float32)-p.astype(np.float32))
    ip,ir,if1=f1_from_features(ci,pi,m)
    _,_,ef1=f1_from_features(ce,pe,m)
    _,sm=ssim(c,p,data_range=255,full=True)
    return {**meta,'pixels':int(m.sum()),'mae':float(diff[m].mean()),
            'ssim_local':float(sm[m].mean()),'ink_precision':ip,'ink_recall':ir,
            'ink_f1':if1,'edge_f1':ef1,
            'false_inserted_ink':int((pi&~ci&m).sum()),
            'missing_ink':int((ci&~pi&m).sum())}

inventory_path=AUDIT/'output_inventory.csv'
if not inventory_path.exists():
    raise RuntimeError('Complete output inventory is required before truth scoring.')
outputs=pd.read_csv(inventory_path)
expected=len(regions)*5*len(RESTORERS)
if len(outputs)!=expected:
    raise RuntimeError(f'Incomplete output inventory: {len(outputs)} expected {expected}')

key_cols=['region_id','arm','restorer']
if outputs.duplicated(key_cols).any():
    raise RuntimeError('Output inventory contains duplicate region/arm/restorer rows.')

metric_path=TABLES/'paired_region_metrics.csv'
existing_metrics=pd.read_csv(metric_path) if metric_path.exists() else pd.DataFrame()
if len(existing_metrics) and existing_metrics.duplicated(key_cols).any():
    raise RuntimeError('Existing metric CSV contains duplicate region/arm/restorer rows.')
done=set(zip(existing_metrics.region_id.astype(str),existing_metrics.arm.astype(str),existing_metrics.restorer.astype(str))) if len(existing_metrics) else set()
rows=existing_metrics.to_dict('records') if len(existing_metrics) else []
region_lookup=regions.set_index('region_id',drop=False)

# Cache masks, crop boxes and clean-derived maps once per region/arm rather than once per output.
region_cache={}
def region_arm_context(region_id,arm,page):
    cache_key=(str(region_id),str(arm))
    if cache_key in region_cache: return region_cache[cache_key]
    r=region_lookup.loc[region_id]
    mask_path=N45/'masks'/str(page)/str(r.region_id).replace(':','_')/f'{arm}.png'
    mask_im=cv2.imread(str(mask_path),0)
    if mask_im is None: raise RuntimeError(f'Unreadable scoring mask: {mask_path}')
    mask=mask_im>0
    box=crop_box(mask)
    clean=page_clean[page]
    if mask.shape!=clean.shape: raise RuntimeError(f'{region_id} {arm}: mask/clean shape mismatch')
    # Compute clean transforms on the padded crop only. They are reused for all restorers.
    if box is None:
        context=(mask,box,None,None)
    else:
        y0,y1,x0,x1=box; clean_crop=clean[y0:y1,x0:x1]
        context=(mask,box,ink(clean_crop),edges(clean_crop))
    region_cache[cache_key]=context
    return context

def score_inventory_row(o):
    region_id=str(o.region_id); page=str(o.page); arm=str(o.arm); restorer=str(o.restorer)
    r=region_lookup.loc[region_id]
    clean=page_clean[page]
    pred=cv2.imread(str(o.output),0)
    if pred is None or pred.shape!=clean.shape:
        raise RuntimeError(f'Invalid scored output: {o.output}')
    mask,box,ci_crop,ce_crop=region_arm_context(region_id,arm,page)
    meta={'region_id':region_id,'page':page,'source_group':o.source_group,
          'arm':arm,'restorer':restorer}
    if box is None:
        return score_one_fast(clean,None,None,pred,mask,box,meta)
    return score_one_fast(clean,ci_crop,ce_crop,pred,mask,box,meta)

# Fail closed if the accelerated path does not reproduce a sample of metrics already
# generated by the original full-page implementation.
if len(existing_metrics):
    sample=existing_metrics.sort_values(key_cols).head(min(EQUIVALENCE_SAMPLE,len(existing_metrics)))
    output_index=outputs.set_index(key_cols,drop=False)
    numeric=['pixels','mae','ssim_local','ink_precision','ink_recall','ink_f1','edge_f1',
             'false_inserted_ink','missing_ink']
    failures=[]
    for _,old in sample.iterrows():
        k=(str(old.region_id),str(old.arm),str(old.restorer))
        if k not in output_index.index:
            failures.append({'key':k,'field':'inventory','old':'present','new':'missing'}); continue
        fresh=score_inventory_row(output_index.loc[k])
        for field in numeric:
            a=old[field]; b=fresh[field]
            same=(pd.isna(a) and pd.isna(b)) or (field in ['pixels','false_inserted_ink','missing_ink'] and int(a)==int(b)) or (field not in ['pixels','false_inserted_ink','missing_ink'] and np.isclose(float(a),float(b),rtol=1e-7,atol=1e-9,equal_nan=True))
            if not same: failures.append({'key':k,'field':field,'old':a,'new':b})
    if failures:
        pd.DataFrame(failures).to_csv(AUDIT/'accelerated_scoring_equivalence_failures.csv',index=False)
        raise RuntimeError('Accelerated scoring equivalence check failed; original CSV remains unchanged.')
    print('Accelerated scoring equivalence check passed:',len(sample),'completed combinations.')

new_since_checkpoint=0
# Region/arm order keeps the same clean/mask context hot for all restorers.
for (region_id,arm),group in outputs.sort_values(['page','region_id','arm','restorer']).groupby(['region_id','arm'],sort=False):
    for _,o in group.iterrows():
        key=(str(o.region_id),str(o.arm),str(o.restorer))
        if key in done: continue
        row=score_inventory_row(o)
        rows.append(row); done.add(key); new_since_checkpoint+=1
        region_cache.pop((str(region_id),str(arm)),None)
    if new_since_checkpoint>=CHECKPOINT_NEW_ROWS:
        atomic_csv(pd.DataFrame(rows),metric_path)
        print('Checkpoint:',len(rows),'/',expected)
        new_since_checkpoint=0

metrics=pd.DataFrame(rows)
if len(metrics)!=expected or metrics.duplicated(key_cols).any():
    atomic_csv(metrics,metric_path)
    raise RuntimeError(f'Metric inventory incomplete or duplicated: {len(metrics)} expected {expected}')
atomic_csv(metrics,metric_path)
print('SCORING COMPLETE:',len(metrics))


In [ ]:
# Whole-page candidate scoring. Clean truth is used only here, after candidate assembly.
whole_candidates=pd.read_csv(TABLES/'whole_page_candidate_inventory.csv')
whole_metric_rows=[]
for _,row in whole_candidates.iterrows():
    page=row.page; clean=page_clean[page]; obs=page_obs[page]; auth=page_auth[page]
    pred=cv2.imread(str(row.candidate),cv2.IMREAD_GRAYSCALE)
    if pred is None or pred.shape!=clean.shape: raise RuntimeError(f'Invalid whole-page candidate: {row.candidate}')
    diff=np.abs(clean.astype(np.float32)-pred.astype(np.float32))
    _,sm=ssim(clean,pred,data_range=255,full=True)
    ip,ir,if1=f1_from_features(ink(clean),ink(pred),auth); ep,er,ef1=f1_from_features(edges(clean),edges(pred),auth)
    whole_metric_rows.append({
        'page':page,'source_group':row.source_group,'arm':row.arm,'restorer':row.restorer,
        'authoritative_pixels':int(auth.sum()),'mae_authoritative':float(diff[auth].mean()) if auth.any() else np.nan,
        'ssim_authoritative':float(sm[auth].mean()) if auth.any() else np.nan,
        'ink_precision':ip,'ink_recall':ir,'ink_f1':if1,'edge_f1':ef1,
        'false_inserted_ink':int((ink(pred)&~ink(clean)&auth).sum()),
        'missing_ink':int((ink(clean)&~ink(pred)&auth).sum()),
        'outside_mask_drift':int(np.sum(pred[~auth]!=obs[~auth])),
        'changed_vs_observed':int(np.sum(pred!=obs)),
    })
whole_page_metrics=pd.DataFrame(whole_metric_rows)
if len(whole_page_metrics)!=len(whole_candidates): raise RuntimeError('Whole-page metric inventory incomplete.')
atomic_csv(whole_page_metrics,TABLES/'whole_page_candidate_metrics.csv')
atomic_csv(whole_page_metrics.groupby(['arm','restorer'],as_index=False).agg(
    pages=('page','nunique'),mae_authoritative=('mae_authoritative','mean'),ssim_authoritative=('ssim_authoritative','mean'),
    ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),
    missing_ink=('missing_ink','sum'),outside_mask_drift=('outside_mask_drift','sum')
),TABLES/'whole_page_candidate_summary.csv')
print('WHOLE-PAGE SCORING COMPLETE:',len(whole_page_metrics))

In [ ]:
# Paired mask effects and oracle-normalised recovery by restorer.
wide=metrics.pivot_table(index=['region_id','page','source_group','restorer'],columns='arm',values=['mae','ink_f1','edge_f1','false_inserted_ink','missing_ink'],aggfunc='first').reset_index();wide.columns=['_'.join([str(x) for x in c if str(x)]) for c in wide.columns]
for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True),('false_inserted_ink',False),('missing_ink',False)]:
 sign=1 if higher else -1;den=sign*(wide[f'{metric}_M4']-wide[f'{metric}_M0']);num=sign*(wide[f'{metric}_M3']-wide[f'{metric}_M0']);wide[f'{metric}_M3_vs_M0_benefit']=num;wide[f'{metric}_oracle_normalised']=np.where(np.abs(den)>1e-9,num/den,np.nan)
wide.to_csv(TABLES/'oracle_normalised_recovery.csv',index=False)
page=metrics.groupby(['page','source_group','arm','restorer'],as_index=False).agg(regions=('region_id','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));page.to_csv(TABLES/'results_by_page.csv',index=False)
group=page.groupby(['source_group','arm','restorer'],as_index=False).agg(pages=('page','nunique'),mae=('mae','mean'),ink_f1=('ink_f1','mean'),edge_f1=('edge_f1','mean'),false_inserted_ink=('false_inserted_ink','sum'),missing_ink=('missing_ink','sum'));group.to_csv(TABLES/'results_by_source_group.csv',index=False)

In [ ]:
# Page-cluster bootstrap for M3 vs M0 by restorer.
rng=np.random.default_rng(SEED);boot_rows=[]
for rest,g in page.groupby('restorer'):
 p=g.pivot(index='page',columns='arm',values=['mae','ink_f1','edge_f1'])
 pages=p.index.to_numpy()
 for metric,higher in [('mae',False),('ink_f1',True),('edge_f1',True)]:
  vals=(p[(metric,'M3')]-p[(metric,'M0')])*(1 if higher else -1);samples=[]
  for _ in range(BOOT):samples.append(float(vals.loc[rng.choice(pages,len(pages),replace=True)].mean()))
  boot_rows.append({'restorer':rest,'metric':metric,'benefit_mean':float(vals.mean()),'ci95_low':float(np.quantile(samples,.025)),'ci95_high':float(np.quantile(samples,.975))})
boot=pd.DataFrame(boot_rows);boot.to_csv(TABLES/'paired_bootstrap_M3_vs_M0.csv',index=False)

In [ ]:
# Diagnostic visual queue.
rank=wide.sort_values('edge_f1_M3_vs_M0_benefit',ascending=False)
for kind,data in [('largest_gains',rank.head(50)),('largest_regressions',rank.tail(50))]:
 for i,(_,q) in enumerate(data.iterrows(),1):
  r=regions[regions.region_id==q.region_id].iloc[0];page=q.page;obs=page_obs[page];clean=page_clean[page];m=45;x0=max(0,int(r.x0)-m);x1=min(obs.shape[1],int(r.x1)+m);y0=max(0,int(r.y0)-m);y1=min(obs.shape[0],int(r.y1)+m)
  paths=[OUTPUTS/page/r.region_id.replace(':','_')/f'M0_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M3_{q.restorer}.png',OUTPUTS/page/r.region_id.replace(':','_')/f'M4_{q.restorer}.png']
  panels=[obs,cv2.imread(str(paths[0]),0),cv2.imread(str(paths[1]),0),cv2.imread(str(paths[2]),0),clean];titles=['observed','M0','M3','M4','clean diagnostic']
  fig,axes=plt.subplots(1,5,figsize=(15,3))
  for ax,im,t in zip(axes,panels,titles):ax.imshow(im[y0:y1,x0:x1],cmap='gray',vmin=0,vmax=255);ax.set_title(t);ax.axis('off')
  plt.tight_layout();plt.savefig(REVIEW/f'{kind}_{i:03d}_{q.region_id.replace(":","_")}_{q.restorer}.png',dpi=150,bbox_inches='tight');plt.close(fig)

In [ ]:
# Fail-closed development verdict.
learned='R3_learned_unet' if ENABLE_LEARNED_RESTORER else 'R1_biharmonic';b=boot[(boot.restorer==learned)&(boot.metric=='edge_f1')]
positive=bool(len(b) and b.iloc[0].ci95_low>0);stable=bool((group[group.restorer==learned].groupby('source_group').size()>0).all())
verdict={'version':'8.6-notebook46','valid_run':True,'mask_lock_id':lock['lock_id'],'primary_restorer':learned,'M3_edge_f1_bootstrap_positive':positive,'source_groups_reported':stable,'decision':'V8_6_POSITIVE_DEVELOPMENT_FINDING' if positive else 'NO_DEMONSTRATED_EASIER_RECONSTRUCTION','production_authority':False}
(OUT/'v8_6_verdict.json').write_text(json.dumps(verdict,indent=2));print(json.dumps(verdict,indent=2))

In [ ]:
artifacts=['source_record.json','restorer_pre_registration.json','v8_6_verdict.json','tables/paired_region_metrics.csv','tables/oracle_normalised_recovery.csv','tables/results_by_page.csv','tables/results_by_source_group.csv','tables/paired_bootstrap_M3_vs_M0.csv','audit/output_inventory.csv','audit/learned_model_inventory.csv','audit/recovery_audit_before_compute.csv','audit/output_completion.json','audit/page_checkpoint_status.csv','tables/whole_page_candidate_inventory.csv','tables/whole_page_candidate_metrics.csv','tables/whole_page_candidate_summary.csv']
rows=[]
for rel in artifacts:
    p=OUT/rel
    if not p.exists():raise RuntimeError(f'Missing upload artefact {rel}')
    dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst)
    rows.append({'relative_path':rel,'bytes':p.stat().st_size,'sha256':sha(p)})
atomic_csv(pd.DataFrame(rows),UPLOAD/'upload_manifest.csv')
(UPLOAD/'README.txt').write_text('V8.6 Notebook 46R6 post-output CPU-only curated review package. Development evidence only.\n')
(UPLOAD/'upload_file_list.txt').write_text('\n'.join(x['relative_path'] for x in rows))
zip_path=UPLOAD/'notebook46R6_upload_bundle.zip'
tmp=UPLOAD/'notebook46R_upload_bundle.tmp.zip'
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
    for p in UPLOAD.rglob('*'):
        if p.is_file() and p not in {zip_path,tmp}:z.write(p,p.relative_to(UPLOAD))
tmp.replace(zip_path)
print('NOTEBOOK 46R6 COMPLETE:',OUT)

In [ ]:
# 46R6 notebook release self-check. This validates the notebook file when available in Colab.
release_record = {
    "notebook_version": NOTEBOOK_VERSION,
    "contract_fingerprint": RUN_CONTRACT_FINGERPRINT,
    "development_only": True,
    "checks": {
        "outside_mask_immutability_required": True,
        "truth_used_only_after_candidate_assembly": True,
        "source_group_separation_required_for_learned_arm": True,
        "checkpoint_reuse_requires_contract_match": True,
        "final_upload_fail_closed": True,
    },
}
release_path = OUT / "46R6_release_record.json" if "OUT" in globals() else Path("46R6_release_record.json")
atomic_write_json(release_path, release_record)
print("Wrote", release_path)
